# nb9.1 — Slide-Timing Simulator: Monte-Carlo of Talk Pacing under Q&A Interruption

*Week 9, Chapter 9.1. Companion notebook.*

You have a 15-minute slot at the symposium and 12 slides you want to land. Last summer two cohorts
of scholars finished the camp with beautiful papers and then got eaten alive at the conference for the
same reason: they ran out of time, skipped the punchline, and stood there confused while the chair pulled
the plug. Sam — who can clock a 7-second mile split in his head — was sure he had pacing under control,
right up until the first questioner ate ninety seconds in slide 4. He never got to his identification
slide. The audience never heard the headline.

The point of this notebook is to make pacing a *quantitative* object you can simulate, not a vibe you
hope to nail. You will (i) lay out a 12-slide plan with budgeted seconds per slide, (ii) draw realistic
**Q&A interruptions** from a synthetic distribution (where on the talk a question lands, and how long it
eats), (iii) Monte-Carlo the whole thing 5,000 times under a fixed seed, and (iv) read off the
**probability you reach the headline slide** and the **probability you finish on time**. We will then
test three interventions every speaker can actually do — trimming background, batching questions for the
end, and a 30-second buffer slide — and see which ones move the dial. The lesson is the one §9.1 promised:
*your headline is the load-bearing slide; protect it with a plan, not a prayer*.

The DGP is synthetic and seeded — no external data, no live APIs — so re-running gives you the same
numbers we report below, and "Your Turn" at the end can rerun with your own slide budget.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless: render to buffers, never to a screen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)  # one generator pinned for the whole notebook

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

print("numpy ", np.__version__)
print("pandas", pd.__version__)
print("scipy ", __import__('scipy').__version__)

## 1. The plan: 12 slides, 15 minutes, with a headline you must reach

Sam's first instinct is to spread 15 minutes evenly: $900 / 12 = 75$ seconds per slide. That is a
recipe for disaster, because slides do not carry equal weight. The title slide takes 20 seconds; the
identification figure — the one a referee will judge the whole paper on — needs at least 90. A pacing
plan is a *budget*, not a uniform partition.

We encode the plan as a pandas DataFrame. The `headline_idx` flags the slide that *must* be reached for
the talk to count as a success: in Sam's case, that is slide 9, the event-study figure that shows
intraday momentum loads on the open and reverses by close. A talk that ends at slide 8 is a *failure*
regardless of how good slides 1–8 were, because the audience leaves without the punchline.

Each slide carries (i) a budgeted seconds figure, (ii) a `min_seconds` floor below which the slide stops
making sense, and (iii) a flag for whether it is the headline. The sum of budgets is 14 minutes, leaving
a 60-second cushion before the 15-minute hard stop.

In [ ]:
# Sam's plan: 12 slides totaling 840 s (14:00), with a 60 s cushion before the 15:00 hard stop.
plan = pd.DataFrame([
    {"slide": 1,  "topic": "Title + framing",         "budget_s":  20, "min_s":  15, "headline": False},
    {"slide": 2,  "topic": "Motivating fact",         "budget_s":  60, "min_s":  40, "headline": False},
    {"slide": 3,  "topic": "Literature 1-sentence",   "budget_s":  45, "min_s":  30, "headline": False},
    {"slide": 4,  "topic": "Data + sample",           "budget_s":  75, "min_s":  50, "headline": False},
    {"slide": 5,  "topic": "Identification design",   "budget_s":  90, "min_s":  60, "headline": False},
    {"slide": 6,  "topic": "Headline DiD table",      "budget_s":  90, "min_s":  60, "headline": False},
    {"slide": 7,  "topic": "Coefficient stability",   "budget_s":  60, "min_s":  45, "headline": False},
    {"slide": 8,  "topic": "Robustness battery",      "budget_s":  60, "min_s":  40, "headline": False},
    {"slide": 9,  "topic": "EVENT STUDY (headline)",  "budget_s": 120, "min_s":  90, "headline": True},
    {"slide": 10, "topic": "Mechanism evidence",      "budget_s":  90, "min_s":  60, "headline": False},
    {"slide": 11, "topic": "Limits + what's next",    "budget_s":  60, "min_s":  40, "headline": False},
    {"slide": 12, "topic": "Closing slide",           "budget_s":  70, "min_s":  45, "headline": False},
])

HARD_STOP_S   = 15 * 60   # 900 s — chair stands up; mic goes off
HEADLINE_SLIDE = int(plan.loc[plan["headline"], "slide"].iloc[0])

print(f"total budgeted    : {plan['budget_s'].sum()} s "
      f"({plan['budget_s'].sum() / 60:.1f} min)")
print(f"hard stop         : {HARD_STOP_S} s (15:00)")
print(f"cushion           : {HARD_STOP_S - plan['budget_s'].sum()} s")
print(f"headline slide    : #{HEADLINE_SLIDE} ({plan.loc[plan['slide'] == HEADLINE_SLIDE, 'topic'].iloc[0]})")
plan

## 2. The interruption DGP: where questions land and how long they eat

A naive Monte-Carlo assumes the talk runs at budget. Real talks do not. A hand goes up — usually around
the data slide, sometimes during identification — and someone asks a clarifying question that eats
anywhere from 20 seconds to two minutes. The pacing risk is not that you talk too long; it is that the
*questions* do.

We model interruptions with two ingredients. First, the **count** of interruptions during the talk is
Poisson with mean $\lambda = 2.5$ (a typical 15-minute finance seminar gets two or three hands during
the talk, plus the formal Q&A at the end). Second, each interruption draws (i) a **slide it lands on**
from a non-uniform distribution that puts most weight on the data and identification slides — that is
where audiences ask — and (ii) a **duration** in seconds from a $\text{LogNormal}$ whose median is 35 s
with a fat right tail that occasionally produces a 90-second derailment. These choices encode the
empirical regularity that *most interruptions are short and most occur early*, which is exactly why
slides 1–6 are the dangerous part of the talk for a tight headline.

In [ ]:
LAMBDA_INTERRUPTIONS = 2.5   # mean # of questions during the talk (Poisson)

# Where questions land. Heavy on data + identification (slides 4-5), light on closing slides.
# Probabilities sum to 1 across all 12 slides.
LAND_WEIGHTS = np.array([
    0.02,  # 1  title
    0.05,  # 2  motivation
    0.05,  # 3  lit
    0.20,  # 4  DATA (clarifying questions hit hardest here)
    0.20,  # 5  IDENTIFICATION
    0.12,  # 6  headline DiD table
    0.08,  # 7  coef stability
    0.06,  # 8  robustness
    0.10,  # 9  event study
    0.06,  # 10 mechanism
    0.04,  # 11 limits
    0.02,  # 12 closing
])
assert np.isclose(LAND_WEIGHTS.sum(), 1.0), "landing weights must sum to 1"

# Interruption duration in seconds: LogNormal with median ~35 s, fat right tail.
# Parameterized so exp(mu) is the median; sigma controls the tail.
MU_LN    = np.log(35.0)
SIGMA_LN = 0.55

# Sanity-check the duration distribution before we Monte-Carlo it.
sample = rng.lognormal(MU_LN, SIGMA_LN, size=10_000)
print("interruption duration (s):")
print(f"  median = {np.median(sample):5.1f}  mean = {sample.mean():5.1f}")
print(f"  p25    = {np.percentile(sample, 25):5.1f}  p75 = {np.percentile(sample, 75):5.1f}")
print(f"  p95    = {np.percentile(sample, 95):5.1f}  p99 = {np.percentile(sample, 99):5.1f}")

The distribution above is the one §9.1 warned you about: the *median* interruption is 35 seconds, which
feels harmless, but one in twenty is over 80 seconds, and one in a hundred is over 120. Those long-tail
events are what kill talks. A Monte-Carlo lets us see how often, given Sam's plan, a long-tail
interruption combines with a heavy-landing slide to push the headline past the 15-minute wall.

### Why a lognormal for durations?

Question times are non-negative, multiplicatively variable (a 20-second question is to a 90-second
question as a 90-second one is to a 6-minute one), and right-skewed: there is no "negative" question, but
there is always the possibility of a long one. The lognormal is the natural family for that — log-time
is roughly normal, time itself is heavy-tailed. We *do not* use a normal: it would give us negative
durations, which are nonsense, and underweight the long tail that actually causes the failures.

## 3. One simulated talk, narrated end-to-end

Before we run 5,000 talks, let us walk through one. The simulation is simple: start a clock at zero;
spend `budget_s` on each slide in order; whenever an interruption hits during a slide, add its duration
to the elapsed time; if at any point the clock crosses the hard stop, the talk *ends mid-slide* — and we
record which slide that was.

The function below returns a per-slide log so you can see what happened: the planned start, the planned
end, the interruption time charged to that slide, and the realized end. That granular log is what makes
the simulator pedagogically honest — you can see *where* the talk went off the rails.

In [ ]:
def simulate_one_talk(plan_df, hard_stop_s, lam, land_w, mu_ln, sigma_ln, gen):
    # Simulate one talk; return per-slide log + summary dict.
    n_int = gen.poisson(lam)
    # Draw interruption landings + durations up front.
    if n_int > 0:
        slides_hit = gen.choice(plan_df["slide"].values, size=n_int, p=land_w)
        durations  = gen.lognormal(mu_ln, sigma_ln, size=n_int)
    else:
        slides_hit = np.array([], dtype=int)
        durations  = np.array([])
    # Aggregate interruption time per slide.
    int_by_slide = pd.Series(0.0, index=plan_df["slide"].values)
    for s, d in zip(slides_hit, durations):
        int_by_slide.loc[s] += d

    log = []
    clock = 0.0
    reached_headline = False
    finished = False
    cut_at_slide = None
    for _, row in plan_df.iterrows():
        slide_budget = row["budget_s"] + int_by_slide.loc[row["slide"]]
        planned_end  = clock + slide_budget
        if planned_end >= hard_stop_s:
            # Talk ends mid-slide.
            cut_at_slide = int(row["slide"])
            log.append({
                "slide": int(row["slide"]), "start_s": clock, "end_s": hard_stop_s,
                "interruption_s": float(int_by_slide.loc[row["slide"]]),
                "cut": True,
            })
            clock = hard_stop_s
            break
        log.append({
            "slide": int(row["slide"]), "start_s": clock, "end_s": planned_end,
            "interruption_s": float(int_by_slide.loc[row["slide"]]),
            "cut": False,
        })
        clock = planned_end
        if row["headline"]:
            reached_headline = True
    else:
        finished = True   # walked all slides without breaking

    log_df = pd.DataFrame(log)
    summary = {
        "n_interruptions":  int(n_int),
        "total_int_s":      float(int_by_slide.sum()),
        "reached_headline": bool(reached_headline),
        "finished":         bool(finished),
        "ended_s":          float(clock),
        "cut_at_slide":     cut_at_slide,
    }
    return log_df, summary

# Walk one example talk and narrate it.
log_one, summary_one = simulate_one_talk(
    plan, HARD_STOP_S, LAMBDA_INTERRUPTIONS, LAND_WEIGHTS, MU_LN, SIGMA_LN, rng
)
print(f"interruptions   : {summary_one['n_interruptions']}  "
      f"(total {summary_one['total_int_s']:.0f} s eaten)")
print(f"reached headline: {summary_one['reached_headline']}")
print(f"finished slate  : {summary_one['finished']}")
print(f"ended at        : {summary_one['ended_s']:.0f} s "
      f"(cut at slide {summary_one['cut_at_slide']})" if summary_one['cut_at_slide']
      else f"ended at        : {summary_one['ended_s']:.0f} s (clean finish)")
log_one.assign(end_min=lambda d: d["end_s"] / 60).round(1)

Notice what happened in that single talk. The interruptions landed where the weights predicted — on the
data and identification slides — and the cumulative clock crept up. Whether Sam reached the headline
depended on the *combination* of how many questions landed and how long each one ran. That dependency
on a tail event is exactly what a Monte-Carlo is built to measure: a single talk is a coin flip, and you
want to know the probability the coin lands "headline reached" over the talk-space.

## 4. Monte-Carlo: 5,000 talks under the baseline plan

Now we run the same simulator 5,000 times under the baseline plan and read off two numbers:

1. **P(reach headline)** — the fraction of simulated talks in which slide 9 (the event study) was
   actually delivered.
2. **P(finish on time)** — the fraction in which all 12 slides ran inside the 15-minute window.

A Monte-Carlo standard error tells us how precise these fractions are at $N = 5{,}000$ draws: for a
proportion $\hat p$, the MC standard error is $\sqrt{\hat p (1-\hat p) / N}$, which is at most
$1 / (2\sqrt N) \approx 0.007$ at $N = 5{,}000$. So the two-decimal-place numbers we print are reliable
to about one percentage point of MC noise — good enough to compare interventions.

In [ ]:
def monte_carlo(plan_df, hard_stop_s, lam, land_w, mu_ln, sigma_ln, gen, n_sims=5000):
    reached = np.zeros(n_sims, dtype=bool)
    finished = np.zeros(n_sims, dtype=bool)
    ended_s = np.zeros(n_sims)
    cut_slides = np.full(n_sims, -1, dtype=int)
    for k in range(n_sims):
        _, s = simulate_one_talk(plan_df, hard_stop_s, lam, land_w, mu_ln, sigma_ln, gen)
        reached[k]  = s["reached_headline"]
        finished[k] = s["finished"]
        ended_s[k]  = s["ended_s"]
        if s["cut_at_slide"] is not None:
            cut_slides[k] = s["cut_at_slide"]
    return reached, finished, ended_s, cut_slides

N_SIMS = 5_000
reached, finished, ended_s, cut_slides = monte_carlo(
    plan, HARD_STOP_S, LAMBDA_INTERRUPTIONS, LAND_WEIGHTS, MU_LN, SIGMA_LN, rng, n_sims=N_SIMS
)

def mc_se(p, n):
    return np.sqrt(p * (1 - p) / n)

p_head = reached.mean()
p_done = finished.mean()
print(f"Baseline plan  ({N_SIMS:,} sims)")
print(f"  P(reach headline)  = {p_head:.3f}   (MC se = {mc_se(p_head, N_SIMS):.3f})")
print(f"  P(finish on time)  = {p_done:.3f}   (MC se = {mc_se(p_done, N_SIMS):.3f})")
print(f"  mean end time      = {ended_s.mean():.0f} s "
      f"({ended_s.mean() / 60:.2f} min)")
print(f"  P(cut mid-talk)    = {(cut_slides >= 0).mean():.3f}")

Read those numbers honestly: under the baseline plan, the headline is reached nearly every time
(about 98%), but the full talk finishes inside 15 minutes only about a third of the time. That is not a
*bad* plan; it is just an *unprotected* one. The reason is the cushion: 60 seconds is not enough slack
to absorb two median interruptions, let alone one long-tail one. Sam reaches his headline event-study
slide in most simulated runs — because it sits at slide 9 and the clock has not yet hit the wall — but
he is being cut off in the back third of the deck (mechanism, limits, closing) in most runs. To see
what is actually killing the talks, look at where the cut lands when one happens.

In [ ]:
# Where did the cut land, when a cut happened?
cut_table = (
    pd.Series(cut_slides[cut_slides >= 0])
      .value_counts()
      .sort_index()
      .rename("count")
      .to_frame()
)
cut_table["share_of_cuts"] = cut_table["count"] / cut_table["count"].sum()
cut_table = cut_table.join(plan.set_index("slide")[["topic"]])
print("Slide where the talk got cut (conditional on being cut):")
print(cut_table.to_string())

The cuts cluster around slides 9–12 — exactly the back half of the talk, which is where the headline
and its mechanism evidence live. That is the diagnostic: the plan is *front-loaded*, and the cushion
sits in slides 11–12, which is the wrong place to put the cushion when the failure mode is "ran out of
time before the headline." Now let us test three plausible interventions.

## 5. Three interventions, each a Monte-Carlo

Every intervention we test corresponds to a *thing a real speaker can do*. We will run each through the
same 5,000-talk Monte-Carlo and compare.

**Intervention A: trim the background.** Cut 15 seconds each from slides 2, 3, and 4. This is the most
common piece of advice an advisor gives: the audience does not need a five-minute literature recap;
they need to know the design and the result.

**Intervention B: batch questions to the end.** Suppress *during-talk* interruptions by setting
$\lambda = 0.5$ instead of 2.5. The speaker says "hold questions until the end" at slide 1 and means it.
This shifts the question-mass to the formal Q&A *after* the talk, where it does not eat the time budget.

**Intervention C: add a buffer slide.** Insert a 30-second "deep breath" slide right before the
headline. The buffer absorbs slippage; if the talk is on time, the speaker uses it to set up the
headline; if it is behind, the speaker skips it and protects the headline.

The three interventions are independent so we can read off which one moves the dial most.

In [ ]:
# --- A: trim background (slides 2, 3, 4) ----------------------------------
plan_A = plan.copy()
plan_A.loc[plan_A["slide"].isin([2, 3, 4]), "budget_s"] = (
    plan_A.loc[plan_A["slide"].isin([2, 3, 4]), "budget_s"] - 15
)
reached_A, finished_A, ended_A, _ = monte_carlo(
    plan_A, HARD_STOP_S, LAMBDA_INTERRUPTIONS, LAND_WEIGHTS, MU_LN, SIGMA_LN, rng, n_sims=N_SIMS
)

# --- B: batch questions to the end (lower lambda) ------------------------
LAMBDA_B = 0.5
reached_B, finished_B, ended_B, _ = monte_carlo(
    plan, HARD_STOP_S, LAMBDA_B, LAND_WEIGHTS, MU_LN, SIGMA_LN, rng, n_sims=N_SIMS
)

# --- C: 30 s buffer slide right before the headline ----------------------
buffer_row = pd.DataFrame([{
    "slide": 8.5, "topic": "Buffer (set up headline)",
    "budget_s": 30, "min_s": 10, "headline": False,
}])
plan_C = pd.concat([plan, buffer_row], ignore_index=True).sort_values("slide").reset_index(drop=True)
plan_C["slide"] = np.arange(1, len(plan_C) + 1)   # re-index slides 1..N
land_C = np.insert(LAND_WEIGHTS, 8, 0.02)   # add small landing weight for the new slide
land_C = land_C / land_C.sum()
reached_C, finished_C, ended_C, _ = monte_carlo(
    plan_C, HARD_STOP_S, LAMBDA_INTERRUPTIONS, land_C, MU_LN, SIGMA_LN, rng, n_sims=N_SIMS
)

# --- Summary table -------------------------------------------------------
summary = pd.DataFrame([
    {"plan": "Baseline",           "P(reach headline)": p_head,                 "P(finish on time)": p_done},
    {"plan": "A: Trim background", "P(reach headline)": reached_A.mean(),       "P(finish on time)": finished_A.mean()},
    {"plan": "B: Batch Qs to end", "P(reach headline)": reached_B.mean(),       "P(finish on time)": finished_B.mean()},
    {"plan": "C: Buffer slide",    "P(reach headline)": reached_C.mean(),       "P(finish on time)": finished_C.mean()},
])
summary["headline_se"] = mc_se(summary["P(reach headline)"], N_SIMS)
summary["finish_se"]   = mc_se(summary["P(finish on time)"], N_SIMS)
print(summary.round(3).to_string(index=False))

The three interventions move the dial in very different ways. *P(reach headline)* is already high
under the baseline, so the four plans cluster near the top; the more revealing column is *P(finish on
time)*, where the differences are dramatic. Trimming the background lifts the finish probability from
about 0.32 to 0.58 — a 26-point improvement — because it gives back 45 seconds of cushion in the front
half of the talk, exactly where interruptions land. Batching questions to the end is the *biggest*
lever: it pushes the finish probability close to 0.88, because $\lambda = 0.5$ means most simulated
talks have zero or one interruption to deal with. The buffer slide is the most instructive result — it
*lowers* the finish probability to about 0.17, because it adds 30 seconds of budget *and* 30 seconds of
opportunity for interruption, on top of an already tight cushion. Once you have run the Monte-Carlo,
"always add a buffer slide" stops being obvious advice.

## 6. The figure: distribution of end times under each plan

A summary table tells you the headline probability; the *distribution* tells you why. We plot the
distribution of realized end times under each plan, with the 15-minute hard stop marked. A plan whose
mass sits comfortably to the left of the wall is safe; a plan whose right tail crashes against the wall
is the one that loses headlines.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(8 * 60, 18 * 60, 41)
styles = [
    ("Baseline",           ended_s,  "0.30", "-"),
    ("A: Trim background", ended_A,  "0.50", "--"),
    ("B: Batch Qs to end", ended_B,  "0.10", "-."),
    ("C: Buffer slide",    ended_C,  "0.65", ":"),
]
for label, arr, color, ls in styles:
    ax.hist(arr / 60, bins=bins / 60, histtype="step", linewidth=1.8,
            color=color, linestyle=ls, label=label)

ax.axvline(15.0, color="black", linewidth=1.2)
ax.text(15.05, ax.get_ylim()[1] * 0.92, "15:00 hard stop",
        fontsize=9, color="black", va="top")
ax.set_xlabel("Realized end time (minutes)")
ax.set_ylabel("Number of simulated talks")
ax.set_title("End-time distribution under each pacing plan (5,000 sims each)")
ax.legend(loc="upper left", framealpha=0.95)
fig.tight_layout()
fig.savefig("/tmp/nb91_endtime_dist.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("end-time distribution figure saved to /tmp/nb91_endtime_dist.png")

Look at the right tails. The baseline distribution has noticeable mass past the 15-minute wall — those
are the talks that get cut. Trim-background pulls the whole distribution about 45 seconds left, but the
right tail is still uncomfortably close to the wall. Batch-questions concentrates the distribution
sharply, with very little mass against the wall — that is why it wins the headline-reach number. The
buffer slide *adds* mean time and shifts the distribution right, which is the opposite of what you
wanted; it would only help if the cut were happening on the closing slide rather than mid-talk.

## 7. The pacing-curve diagnostic: where the clock is at each slide

The other diagnostic worth seeing — and the one most useful to *carry* to your real talk — is the
**pacing curve**: the mean elapsed clock time at the *end* of each slide, with a shaded band for the 5th
and 95th percentiles across simulated talks. If the upper band crosses the hard stop *before* the
headline slide, the plan is broken. If it crosses *after*, the plan is roughly safe, and the gap
between the upper band and the wall at the headline slide is your real cushion.

In [ ]:
def pacing_curve(plan_df, hard_stop_s, lam, land_w, mu_ln, sigma_ln, gen, n_sims=1000):
    n_slides = len(plan_df)
    times = np.zeros((n_sims, n_slides))
    for k in range(n_sims):
        log_df, summary_k = simulate_one_talk(plan_df, hard_stop_s, lam, land_w, mu_ln, sigma_ln, gen)
        # Pad with hard_stop_s if cut mid-talk (the speaker hits the wall there and after).
        ends = log_df["end_s"].values
        if len(ends) < n_slides:
            ends = np.concatenate([ends, np.full(n_slides - len(ends), hard_stop_s)])
        times[k, :] = ends
    return times

times = pacing_curve(plan, HARD_STOP_S, LAMBDA_INTERRUPTIONS, LAND_WEIGHTS, MU_LN, SIGMA_LN, rng, n_sims=1000)
mean_t = times.mean(axis=0)
p05 = np.percentile(times, 5, axis=0)
p95 = np.percentile(times, 95, axis=0)
slides = plan["slide"].values

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(slides, p05 / 60, p95 / 60, color="0.80", label="5-95% band")
ax.plot(slides, mean_t / 60, "-o", color="black", linewidth=1.6, label="mean clock at end of slide")
ax.axhline(15.0, color="0.20", linewidth=1.2, label="15:00 hard stop")
ax.axvline(HEADLINE_SLIDE, color="0.40", linestyle="--", linewidth=1.0,
           label=f"headline slide #{HEADLINE_SLIDE}")
ax.set_xlabel("Slide number")
ax.set_ylabel("Elapsed clock at end of slide (minutes)")
ax.set_title("Pacing curve under the baseline plan (1,000 sims)")
ax.set_xticks(slides)
ax.legend(loc="upper left", framealpha=0.95)
fig.tight_layout()
fig.savefig("/tmp/nb91_pacing_curve.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("pacing-curve figure saved to /tmp/nb91_pacing_curve.png")
print(f"\nat the headline slide (#{HEADLINE_SLIDE}): mean clock = {mean_t[HEADLINE_SLIDE-1]/60:.2f} min, "
      f"p95 = {p95[HEADLINE_SLIDE-1]/60:.2f} min")

The pacing curve makes the issue concrete: by slide 9 (the headline), the mean elapsed time is just
under 12 minutes and the upper band sits around 14:20 — close to the wall but still safely on the right
side of it in most simulated runs. The trouble starts in slides 10–12, where the upper band crashes
through the 15-minute line and forces cuts. The intervention that helps is the one that drags the
*upper band* down, not the one that drags the mean down. That is why batch-questions wins over
buffer-slide: it shaves the right tail of the interruption-time distribution, which is exactly the
thing the upper band reflects.

## 8. The point of the exercise

The pacing simulator is small — under 200 lines of code — but it makes a discipline visible. Sam stopped
arguing about pacing in vibes and started arguing about *distributions*: where the questions land, how
long the long tail eats, how many seconds of cushion the headline needs. The conversation with his
advisor went from "you'll be fine, just practice" to "your upper-band crosses the wall at slide 9; either
trim the front half or ask the chair to enforce questions at the end."

A few habits to carry forward:

- **Name your headline slide before you build the deck.** If you cannot point to *the* slide that, if
  reached, makes the talk a success, you have not yet decided what the talk is about.
- **Budget seconds, not slides.** A 12-slide deck is not 75 s × 12; it is whatever the *content* of
  each slide is worth. The cushion goes in the right place — before the headline — not on the closing
  slide where it cannot save you.
- **Treat interruptions as a distribution, not a worst-case.** The median question is short; the long
  tail is what kills you. Plan for the 95th percentile, not the average.

The exact symposium-day pacing protocol — how to ask the chair to hold questions, how to recover from a
long-tail interruption mid-talk, what to skip if you are 90 seconds behind at slide 6 — lives in the
chapter prose.

---

### Your Turn

1. **Change the headline slide.** Move `headline = True` from slide 9 to slide 7 in `plan` and re-run
   the baseline Monte-Carlo. Why does P(reach headline) jump so much? What does that say about *where*
   in the deck you should put the most important figure?
2. **Stress-test interruption load.** Re-run the baseline with $\lambda = 4$ (a chatty audience) and
   $\lambda = 1$ (a quiet one). At what $\lambda$ does the baseline plan stop reaching the headline 80%
   of the time? Express the answer as a one-sentence rule a speaker can act on.
3. **Compute a real-money cushion.** What is the minimum *total cushion* (sum of `budget_s` minus 14
   minutes) that keeps P(reach headline) above 0.95 under baseline $\lambda$? Try total cushions of
   60, 90, 120, and 180 seconds, redistributed evenly across slides 2–8, and report the resulting
   probabilities. This is the quantitative version of "leave yourself room.\"